# Build Your AI Clone

In this workshop you'll analyze your own WhatsApp chat with Python and build an AI that texts like you.

**How this notebook works**

- It's made of **cells**. Grey cells have code, white cells (like this one) have explanations.
- To run a code cell, click on it and press **Shift + Enter**. Run them in order, from top to bottom.
- Lines that start with `#` are **comments**: notes for humans that Python ignores.
- Cells marked with **Challenge** are for you to change and run again. You can't break anything!

## Setup

The hard parts of this workshop (reading the WhatsApp format, drawing charts, the chat window...) are in a separate file called `workshop.py`. This cell downloads it and uses **`import`** to bring all its tools into the notebook, so we can use them without writing them ourselves. That's how programmers work: we reuse code instead of writing everything from zero.

It also installs **Gradio**, a library to make little web apps. It takes about 30 seconds.

In [ ]:
!pip install -q gradio
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/MateoNeiraCastillo/coding-club-workshops/main/01-ai-clone/workshop.py", "workshop.py")
from workshop import *

print("Ready!")

## Step 1 · Get your chat

First, export one of your WhatsApp chats **without media**:

- **iPhone:** open the chat → tap the name at the top → **Export Chat** → **Without Media**
- **Android:** open the chat → ⋮ → **More** → **Export chat** → **Without media**

Send the file to your laptop. Both `.txt` and `.zip` files work.

Don't want to use your own chat? Type `y` when asked and you'll get **"Mentors Anonymous"**, a group chat between **Tony Stark, Yoda and Haymitch** (all the messages are original, written for this workshop).

**New concept: functions and variables.** `get_chat()` is a **function**: a piece of code with a name that does a job. This one asks you for the chat and **returns** its text. With `=` we save that text in a **variable** called `chat_text`, so we can use it later.

In [ ]:
chat_text = get_chat()

## Step 2 · Read the chat

**New concept: lists and dictionaries.** `read_chat` turns the text into a **list** of messages. A list is a group of things in order, written between `[ ]`.

Each message is a **dictionary**: a group of values, each one with a name (a **key**). Every message has three keys: `"date"`, `"author"` and `"text"`.

In [ ]:
messages = read_chat(chat_text)
show_messages(messages)

You get one item of a list with its position between `[ ]`. Python starts counting at **0**, so `messages[0]` is the first message. Then you get one value of a dictionary with its key: `first_message["author"]`.

Try changing the `0` to `1`, `2`... or to `-1` (the last message!) and run the cell again.

In [ ]:
first_message = messages[0]

print(first_message["author"])
print(first_message["text"])

## Step 3 · Who are you?

**New concept: loops and conditions.** A `for` **loop** repeats the same code for every item of a list. Here we go through every message and, **if** its author is not in our list yet, we add them with `.append()`.

Notice the spaces at the start of the lines: they tell Python which code is inside the loop and which is inside the `if`.

In [ ]:
authors = []

for message in messages:
    if message["author"] not in authors:
        authors.append(message["author"])

show_people(authors)

Now type your number. `input()` asks you something and `int()` turns your answer into a number.

We do `choice - 1` because the list starts counting at 0. Then we use another loop to keep only **your** messages.

The `f` before the quotes makes an **f-string**: whatever is inside `{ }` gets replaced by its value.

In [ ]:
choice = int(input("Which number is you? "))
my_name = authors[choice - 1]

my_messages = []
for message in messages:
    if message["author"] == my_name:
        my_messages.append(message)

card(f"Hi {my_name}!", f"You sent {len(my_messages)} messages in this chat.")

## Step 4 · Your WhatsApp Wrapped

**New concept: counting with `Counter`.** A `Counter` is a special dictionary for counting things. Every time we do `+= 1`, we add one to that person's count. Then `.most_common(10)` gives us the 10 biggest counts.

In [ ]:
message_counts = Counter()

for message in messages:
    message_counts[message["author"]] += 1

bar_chart(message_counts.most_common(10), "Who texts the most", color=RED)

Same idea, but now we count **your words**. There are two loops, one inside the other: for every message, and then for every word in that message. We skip boring words like "the" or "que" (they're in `STOP_WORDS`).

In [ ]:
word_counts = Counter()

for message in my_messages:
    for word in get_words(message["text"]):
        if word not in STOP_WORDS:
            word_counts[word] += 1

bar_chart(word_counts.most_common(10), "Your top 10 words")

### Challenge 1

In the cell above, change it to show your **top 5** words instead of 10, and change its color by adding `, color=RED` after the title (or any color like `"#00B894"`). Then run it again.

### Challenge 2

How many times do you say a word? Change `"hello"` to any word or name and run the cell.

In [ ]:
search = "hello"

times = 0
for message in my_messages:
    if search in message["text"].lower():
        times += 1

print(f'You wrote "{search}" in {times} messages')

## Step 5 · Pick the messages for your clone

Your clone learns from real messages you sent. First we save 5 real moments for the game at the end (your clone will never see them). Then you choose how many messages it learns from (up to 1000): more messages usually make a better clone.

`random.sample` picks messages at random, and `min()` gives the smallest number, so we never ask for more messages than you have.

**These messages will be sent to Google's AI (Gemini).** On the free plan, Google may keep them and people at Google may read them.

In [ ]:
game_moments, candidates = save_game_moments(messages, my_name)
print(f"You have {len(candidates)} messages with 3 or more words.")

amount = int(input("How many do you want your clone to learn from? (up to 1000): "))
amount = min(amount, 1000, len(candidates))
examples = random.sample(candidates, amount)

To remove messages you don't want to share, type words they contain (names, topics...). Every message with one of those words is removed.

`.split(",")` cuts your answer into a list at every comma, and `.strip()` removes the spaces around each word.

In [ ]:
answer = input("Type words to remove, separated by commas, or just press Enter: ")

blocked_words = []
for word in answer.lower().split(","):
    if word.strip() != "":
        blocked_words.append(word.strip())

safe_examples = []
for text in examples:
    is_safe = True
    for word in blocked_words:
        if word in text.lower():
            is_safe = False
    if is_safe:
        safe_examples.append(text)

examples = safe_examples
card(f"{len(examples)} messages will be sent to the AI", "Your clone will learn your style from them.")

if input("Do you want to see them? (y/n): ").lower() == "y":
    for text in examples:
        print("-", text)

## Step 6 · Your Style DNA

Now we measure how you text: your signature phrase, how fast you reply, who starts the conversations... The AI also reads the messages you picked and describes your vibe.

`make_style_dna` returns everything in a **dictionary** called `style`.

In [ ]:
style = make_style_dna(messages, my_name, examples)
show_style_dna(style, my_name)

### Challenge 3

Because `style` is a dictionary, you can print any value with its key. Try some of these: `"vibe"`, `"signature_phrase"`, `"reply_speed"`, `"favorite_laugh"`, `"starts_lowercase"`, `"top_words"`.

In [ ]:
print(style["reply_speed"])

## Step 7 · Create your AI clone

An AI like Gemini doesn't know you. To make it text like you, we write a **prompt**: a long text with your Style DNA, your example messages and some rules. Then we send it to the AI with `ai.generate_text()`.

**New concept: making your own function.** With `def` we create a function called `clone_reply`. It gets a message, builds the prompt and returns the AI's answer. We'll use it for the chat and the game.

In [ ]:
from google.colab import ai

extra_rules = ""

def clone_reply(new_message, history=[]):
    prompt = build_prompt(my_name, style, examples, extra_rules, history, new_message)
    return ai.generate_text(prompt).strip()

print(clone_reply("hey what are you doing?"))

Curious about what we actually send to the AI? This prints the prompt (with only 5 example messages, so it isn't too long).

In [ ]:
print(build_prompt(my_name, style, examples[:5], extra_rules, [], "hey what are you doing?"))

### Challenge 4

Give your clone a secret rule! In the `clone_reply` cell, write something inside `extra_rules`, like `"Always mention pizza"` or `"Answer like a pirate"`. Run that cell again and see how the answer changes. (Set it back to `""` before the game!)

## Step 8 · Chat with your clone

Run this cell and open the link that ends in `.gradio.live`. You can send it to your friends: it works for 72 hours, while this notebook is open.

In [ ]:
start_chat(clone_reply, my_name)

## Step 9 · Real or Clone?

Time to test your clone. Remember the 5 real moments we saved in Step 5? For each one, your clone answers the same message, and we mix its answer with your real one.

First prepare the rounds. It takes about a minute, because the AI writes 5 answers.

In [ ]:
rounds = prepare_game(game_moments, clone_reply)

Now play! Open the `.gradio.live` link and give it to the person next to you (or send it to a friend). Can they tell which reply is really yours?

In [ ]:
start_game(rounds, my_name)

## What you learned today

Variables, functions, lists, dictionaries, `for` loops, `if` conditions, f-strings, counting with `Counter`, and how to talk to an AI with a prompt. That's real Python!

Want to keep going? Come to the next IE Coding Club workshop.